### Загрузка данных

In [ ]:
import polars as pl

In [ ]:
df = pl.read_parquet("data/train_part_1.parquet")
df_labels = pl.read_parquet("data/train_labels.parquet")
df = df.join(
    df_labels,
    on=["customer_id", "event_id"],
    how="left"
)

In [ ]:
# Вывод общей информации
print(df.shape)
print(df.columns)

for name, dtype in zip(df.columns, df.dtypes):
    print(f"{name}: {dtype}")

(28618594, 26)
['customer_id', 'event_id', 'event_dttm', 'event_type_nm', 'event_desc', 'channel_indicator_type', 'channel_indicator_sub_type', 'operaton_amt', 'currency_iso_cd', 'mcc_code', 'pos_cd', 'accept_language', 'browser_language', 'timezone', 'session_id', 'operating_system_type', 'battery', 'device_system_version', 'screen_size', 'developer_tools', 'phone_voip_call_state', 'web_rdp_connection', 'compromised', 'target', 'hour', 'weekday']
customer_id: Int64
event_id: Int64
event_dttm: String
event_type_nm: Int32
event_desc: Int32
channel_indicator_type: Int32
channel_indicator_sub_type: Int32
operaton_amt: Float64
currency_iso_cd: Int32
mcc_code: String
pos_cd: Int32
accept_language: String
browser_language: String
timezone: Int32
session_id: Int64
operating_system_type: Int32
battery: String
device_system_version: String
screen_size: String
developer_tools: String
phone_voip_call_state: Int32
web_rdp_connection: Int32
compromised: String
target: Int32
hour: Int8
weekday: Int8

In [ ]:
# Проверка наличия пропусков
df.select(pl.all().null_count())

customer_id,event_id,event_dttm,event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,accept_language,browser_language,timezone,session_id,operating_system_type,battery,device_system_version,screen_size,developer_tools,phone_voip_call_state,web_rdp_connection,compromised,target
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,13353367,12996950,20440094,26134680,25452822,25968210,25867893,11400983,25917436,25968210,19046803,19055299,19888868,22639583,24055569,15673439,28589128


Примерно 70% признаков имеет большое количество пропусков (40% всех значений и более). 

In [5]:
df.group_by("target").len().with_columns(
    (pl.col("len") / pl.col("len").sum()).alias("ratio")
)

target,len,ratio
i32,u32,f64
null,28589128,0.99897
0,12082,0.000422
1,17384,0.000607


### Денежные суммы в нормальных и мошеннических операциях

In [7]:
# Распределение суммы операции
df.select("operaton_amt").describe()

statistic,operaton_amt
str,f64
"""count""",1.5265227e7
"""null_count""",1.3353367e7
"""mean""",2.4512e6
"""std""",2.1265e8
"""min""",0.0
"""25%""",24019.0
"""50%""",71252.0
"""75%""",278096.0
"""max""",2.7685e11


In [ ]:
# Сравнение сумм мошеннических и нормальных операций
df.group_by("target").agg([
    pl.col("operaton_amt").mean().alias("mean_amt"),
    pl.col("operaton_amt").median().alias("median_amt")                                     
]).sort("target")

target,mean_amt,median_amt
i32,f64,f64
null,2.4393e6,71137.0
0,2.2886e7,1.01e6
1,7.0426e6,362916.0


### Категориальные признаки и доля мошеннических операций

In [ ]:
# MCC-код
df.group_by("mcc_code").agg([
    pl.len(),
    pl.col("target").mean().alias("fraud_rate")
]).sort("fraud_rate", descending=True)

mcc_code,len,fraud_rate
str,u32,f64
"""16""",180184,0.984772
"""3""",80808,0.96
"""19""",321235,0.843188
"""13""",25565,0.833333
"""12""",11957,0.833333
"""18""",90616,0.811429
"""10""",899120,0.653324
"""14""",211199,0.65
"""15""",781895,0.56383


In [ ]:
# Тип операции
df.group_by("event_type_nm").agg([
    pl.len(),
    pl.col("target").mean().alias("fraud_rate")
]).sort("fraud_rate", descending=True)

event_type_nm,len,fraud_rate
i32,u32,f64
1,30,null
5,3128,null
12,3917,0.784314
6,28784,0.704762
0,19933,0.655172
14,11657184,0.650647
15,16205,0.627027
7,11945911,0.5909
3,246317,0.58805


### Временные паттерны

Предположим, что количество мошеннических операций увеличивается в ночное время суток и выходные дни.

In [21]:
df = df.with_columns([
    pl.col("event_dttm").str.strptime(pl.Datetime, '%Y-%m-%d %H:%M:%S').dt.hour().alias("hour"),
    pl.col("event_dttm").str.strptime(pl.Datetime, '%Y-%m-%d %H:%M:%S').dt.weekday().alias("weekday")
])

In [ ]:
# Время суток и доля мошеннических операций
pl.Config.set_tbl_rows(24)

df.group_by("hour").agg([
    pl.col("target").mean().alias("fraud_rate"),
    pl.len()
]).sort("fraud_rate", descending=True)

hour,fraud_rate,len
i8,f64,u32
22,0.816092,187226
23,0.80427,197110
21,0.760766,247289
0,0.753521,257175
20,0.737589,357256
1,0.674757,357220
19,0.665087,563488
17,0.658895,1123526
16,0.64579,1398657


In [23]:
# Дни недели и доля мошеннических операций
df.group_by("weekday").agg([
    pl.col("target").mean().alias("fraud_rate"),
    pl.len()
]).sort("fraud_rate", descending=True)

weekday,fraud_rate,len
i8,f64,u32
6,0.629252,3826491
7,0.611089,3105836
4,0.593615,4316576
3,0.58563,4279693
5,0.58525,4698847
2,0.582793,4286272
1,0.56268,4104879


Предположим, что для мошенников характерны быстрые серии операций и рассмотрим время между операциями.

In [ ]:
df = df.with_columns([
    (pl.col("event_dttm") - pl.col("event_dttm").shift(1))
    .over("customer_id")
    .dt.total_seconds()
    .alias("time_diff")
])

In [ ]:
df.group_by("target").agg([
    pl.col("time_diff").mean().alias("mean_time_diff"),
    pl.col("time_diff").median().alias("median_time_diff")
])

### Признаки, часто указывающие на мошеннические операции

Рассмотрим признаки web_rdp_connection - флаг наличия удаленного управления над устройством, compromised - наличие Root-доступа на устройстве, developer_tools - настройки разработчика (флаг на устройстве), phone_voip_call_state - флаг события VoIP-звонка во время проведения операции. Предположим, что наличие этих флагов часто указывает на мошенническую операцию.

In [ ]:
fraud_flags = [
    "web_rdp_connection",
    "compromised",
    "developer_tools",
    "phone_voip_call_state"
]

for col in fraud_flags:
    print(
        df.group_by(col)
        .agg(pl.col("target").mean().alias("fraud_rate"))
    )

shape: (3, 2)
┌────────────────────┬────────────┐
│ web_rdp_connection ┆ fraud_rate │
│ ---                ┆ ---        │
│ i32                ┆ f64        │
╞════════════════════╪════════════╡
│ null               ┆ 0.607014   │
│ 0                  ┆ 0.520615   │
│ 1                  ┆ 0.222749   │
└────────────────────┴────────────┘
shape: (3, 2)
┌─────────────┬────────────┐
│ compromised ┆ fraud_rate │
│ ---         ┆ ---        │
│ str         ┆ f64        │
╞═════════════╪════════════╡
│ null        ┆ 0.621731   │
│ 0           ┆ 0.551877   │
│ 1           ┆ 0.166667   │
└─────────────┴────────────┘
shape: (3, 2)
┌─────────────────┬────────────┐
│ developer_tools ┆ fraud_rate │
│ ---             ┆ ---        │
│ str             ┆ f64        │
╞═════════════════╪════════════╡
│ null            ┆ 0.623602   │
│ 0               ┆ 0.511211   │
│ 1               ┆ 0.488571   │
└─────────────────┴────────────┘
shape: (3, 2)
┌───────────────────────┬────────────┐
│ phone_voip_call_state

### Поведение клиента

In [27]:
df.group_by("customer_id").agg([
    pl.len().alias("num_events"),
    pl.col("target").sum().alias("fraud_count")
]).describe()

statistic,customer_id,num_events,fraud_count
str,f64,f64,f64
"""count""",33333.0,33333.0,33333.0
"""null_count""",0.0,0.0,0.0
"""mean""",1.2331e14,858.566406,0.521525
"""std""",1.1052e11,427.691498,1.696275
"""min""",1.2312e14,45.0,0.0
"""25%""",1.2322e14,521.0,0.0
"""50%""",1.2331e14,834.0,0.0
"""75%""",1.2341e14,1148.0,0.0
"""max""",1.2350e14,3376.0,50.0
